# REHAB24-6 MMPose / RTMPose 2D skeleton features (Colab GPU)

Third estimated-skeleton comparison group, alongside the Vicon mocap baseline and the MediaPipe pseudo-3D features. MMPose/RTMPose gives **more accurate 2D keypoints but no learned depth**, so this runs the *same* geometric pipeline as the MediaPipe extractor on the 2D image branch only (feature dim 1188 vs MediaPipe's 2970).

Unlike MediaPipe (CPU-only), RTMPose genuinely uses the GPU — so use a **GPU runtime** (`Runtime > Change runtime type > GPU`).

## Before you run
Upload the repo folder (including `data/REHAB24-6/`) to your Google Drive, e.g. `MyDrive/x-coach/`. The `data/REHAB24-6/processed/manifest.csv` + `splits/` must be present (already built locally). Then run the cells top to bottom. The extractor is **resumable** (it skips reps whose `.npz` already exists), so if Colab disconnects you can re-run the same cells and it continues.

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Install the pose runtime, then make onnxruntime use the GPU.
#    rtmlib pulls in the CPU `onnxruntime`; it and `onnxruntime-gpu` CONFLICT — when
#    both are present the CPU build wins and CUDA vanishes from the providers list.
#    So: install rtmlib, then drop BOTH onnxruntime packages and reinstall only the GPU one.
!pip -q install rtmlib
!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install onnxruntime-gpu
print('\n>>> Now RESTART the session: Runtime > Restart session.')
print('>>> Then run from cell 3 onward (skip cells 1-2; the Drive mount survives a restart).')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.7 MB/s eta 0:00:00

>>> Now RESTART the session: Runtime > Restart session.
>>> Then run from cell 3 onward (skip cells 1-2; the Drive mount survives a restart).


In [1]:
# 3. Confirm the GPU is visible to both the driver and onnxruntime.
#    Run this AFTER the cell-2 restart. If CUDA is still missing here, the kernel
#    didn't pick up the new lib — restart the session once more and re-run this cell.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import onnxruntime as ort
providers = ort.get_available_providers()
print('onnxruntime:', ort.__version__, '| providers:', providers)
assert 'CUDAExecutionProvider' in providers, (
    'No CUDA provider. Re-run cell 2 (the onnxruntime-gpu swap), then '
    'Runtime > Restart session, then run this cell again.'
)
print('GPU ready.')

Tesla T4, 15360 MiB
onnxruntime: 1.26.0 | providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
GPU ready.


In [2]:
# 4. Paths. REPO_ON_DRIVE = where you uploaded the repo (with data/REHAB24-6 inside).
import sys, shutil, time
from pathlib import Path

REPO_ON_DRIVE = Path('/content/drive/MyDrive/x-coach')          # <-- adjust if you uploaded elsewhere
LOCAL_DATA   = Path('/content/REHAB24-6')                        # fast local copy for video decode

assert REPO_ON_DRIVE.exists(), f'Repo not found on Drive: {REPO_ON_DRIVE}'
sys.path.insert(0, str(REPO_ON_DRIVE))                            # so `import src...` works

# Copy data/REHAB24-6 from Drive to local disk (Drive FUSE is slow for heavy video IO).
src_data = REPO_ON_DRIVE / 'data' / 'REHAB24-6'
assert (src_data / 'processed' / 'manifest.csv').exists(), 'manifest.csv missing — build it locally first.'
if not LOCAL_DATA.exists():
    print('Copying videos to local disk (one-time, a few minutes for ~7.8GB)...')
    t = time.time(); shutil.copytree(src_data, LOCAL_DATA)
    print(f'  copied in {time.time()-t:.0f}s')
else:
    print('Local copy already present:', LOCAL_DATA)
print('videos:', len(list(LOCAL_DATA.rglob('*.mp4'))))

Copying videos to local disk (one-time, a few minutes for ~7.8GB)...
  copied in 426s
videos: 130


In [3]:
# 5. Smoke test: extract ONE video first to validate the runtime end-to-end.
from src.rehab24.mmpose_skeleton_features import extract_features_for_manifest

MANIFEST   = LOCAL_DATA / 'processed' / 'manifest.csv'
OUTPUT_DIR = LOCAL_DATA / 'processed' / 'mmpose_skeleton_features'

written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=OUTPUT_DIR,
    runtime='rtmlib', model='balanced', device='cuda:0', video_limit=1,
)
print('smoke-test reps written:', written)

import numpy as np
sample = next(OUTPUT_DIR.rglob('*.npz'))
with np.load(sample) as d:
    f = d['video_feature']
print('sample', sample.name, '| feature_dim', f.shape[0], '| finite', bool(np.isfinite(f).all()))

smoke-test reps written: 0
sample Ex3_PM_107_rep5_cam17.npz | feature_dim 1188 | finite True


In [4]:
# 6. Full extraction (all 130 videos). Resumable: re-run this cell after any disconnect.
#    rtmlib 'balanced' on a T4 is roughly a few hours for the full set.
written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=OUTPUT_DIR,
    runtime='rtmlib', model='balanced', device='cuda:0',
)
print('reps written this run:', written)
n = len(list(OUTPUT_DIR.rglob('*.npz')))
print('total .npz now:', n, '(expect 2144)')

Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip" to /root/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.zip
100%|██████████| 89.9M/89.9M [00:02<00:00, 39.5MB/s]
Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmw/onnx_sdk/rtmw-dw-x-l_simcc-cocktail14_270e-256x192_20231122.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmw-dw-x-l_simcc-cocktail14_270e-256x192_20231122.zip


load /root/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend


100%|██████████| 203M/203M [00:05<00:00, 39.6MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmw-dw-x-l_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[1/10] rtmlib on Ex6/PM_043-Camera17-30fps.mp4 (20 reps)...


/content/drive/MyDrive/x-coach/src/rehab24/skeleton_features.py:62: RuntimeWarning: Mean of empty slice
  np.nanmean(flat, axis=0),
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/content/drive/MyDrive/x-coach/src/rehab24/skeleton_features.py:64: RuntimeWarning: All-NaN slice encountered
  np.nanmin(flat, axis=0),
/content/drive/MyDrive/x-coach/src/rehab24/skeleton_features.py:65: RuntimeWarning: All-NaN slice encountered
  np.nanmax(flat, axis=0),
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


[2/10] rtmlib on Ex6/PM_043-Camera18-30fps-transposed.mp4 (20 reps)...
[3/10] rtmlib on Ex6/PM_105-Camera17-30fps.mp4 (21 reps)...
[4/10] rtmlib on Ex6/PM_105-Camera18-30fps-transposed.mp4 (21 reps)...
[5/10] rtmlib on Ex6/PM_113-Camera17-30fps.mp4 (22 reps)...
[6/10] rtmlib on Ex6/PM_113-Camera18-30fps-transposed.mp4 (22 reps)...
[7/10] rtmlib on Ex6/PM_118-Camera17-30fps.mp4 (23 reps)...
[8/10] rtmlib on Ex6/PM_118-Camera18-30fps-transposed.mp4 (23 reps)...
[9/10] rtmlib on Ex6/PM_126-Camera17-30fps.mp4 (20 reps)...
[10/10] rtmlib on Ex6/PM_126-Camera18-30fps-transposed.mp4 (20 reps)...
reps written this run: 212
total .npz now: 2144 (expect 2144)


In [5]:
# 7. Verify all reps are present and finite
import numpy as np
paths = list(OUTPUT_DIR.rglob('*.npz'))
bad = [p.name for p in paths if not np.isfinite(np.load(p)['video_feature']).all()]
print(f'total={len(paths)}  non-finite={len(bad)}')
print('OK' if len(paths) == 2144 and not bad else f'CHECK: {bad[:5]}')

total=2144  non-finite=0
OK


In [6]:
# 8. Copy the (small) feature dir back to Drive so you can pull it locally for LOSO.
dst = REPO_ON_DRIVE / 'data' / 'REHAB24-6' / 'processed' / 'mmpose_skeleton_features'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(OUTPUT_DIR, dst)
import subprocess
size = subprocess.run(['du','-sh',str(dst)], capture_output=True, text=True).stdout.strip()
print('copied features to Drive:', size)

copied features to Drive: 11M	/content/drive/MyDrive/x-coach/data/REHAB24-6/processed/mmpose_skeleton_features


## Back on your local machine
Pull the feature dir from Drive into the repo, then train + cross-validate:
```bash
# place it at: data/REHAB24-6/processed/mmpose_skeleton_features/
source .venv/bin/activate

# single fixed-split run (matches the other groups)
python scripts/rehab24/train_correctness_classifier.py \
  --feature-dir data/REHAB24-6/processed/mmpose_skeleton_features

# the trustworthy yardstick: LOSO mean±std
python scripts/rehab24/loso_cross_validation.py \
  --feature-dir data/REHAB24-6/processed/mmpose_skeleton_features \
  --summary-output data/REHAB24-6/processed/correctness_loso_mmpose.json
```